# 02. Distribution Fitting

This notebook fits candidate flood-frequency models using **BestFit model objects** and compares likelihood-based fit quality.

Primary APIs used here are from `RMC.BestFit`: `DataFrame`, `ExactData`, and `Model.UnivariateDistribution` methods such as `SetParameterValues` and `LogLikelihood`.

In [ ]:
import pythonnet
pythonnet.load("netfx")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_donet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from RMC.BestFit import DataFrame, ExactData
from RMC.BestFit.Model import UnivariateDistribution
from Numerics.Distributions import UnivariateDistributionType, Normal, Gumbel, GeneralizedExtremeValue

In [ ]:
# Step 1: Build BestFit DataFrame (core BestFit input structure)
rng = np.random.default_rng(12)
years = np.arange(1980, 1980 + 55)
true_flow = 9000 + 40 * (years - years.min())
noise = rng.normal(0, 900, size=len(years))
peaks = np.maximum(500, true_flow + noise)

df = DataFrame()
for y, q in zip(years, peaks):
    df.ExactSeries.Add(ExactData(int(y), float(q)))
df.PlottingParameter = 0.0
df.CalculatePlottingPositions()

x = np.array([df.ExactSeries[i].Value for i in range(df.ExactSeries.Count)], dtype=float)
mu = float(np.mean(x))
sd = float(np.std(x, ddof=1))

# Step 2: Build BestFit models + set parameter vectors (BestFit method)
models = {
    "Normal": (UnivariateDistributionType.Normal, [mu, sd]),
    "Gumbel": (UnivariateDistributionType.Gumbel, [mu, 0.78 * sd]),
    "GEV": (UnivariateDistributionType.GeneralizedExtremeValue, [mu, 0.80 * sd, 0.08]),
}

rows = []
results = {}
for name, (dtype, params) in models.items():
    m = UnivariateDistribution(df, dtype)
    p = convert_to_donet_array(params)
    m.SetParameterValues(p)
    ll = float(m.LogLikelihood(p))
    k = len(params)
    aic = 2 * k - 2 * ll
    bic = np.log(len(x)) * k - 2 * ll
    rows.append({"model": name, "log_likelihood": ll, "AIC": aic, "BIC": bic})
    results[name] = m

fit_table = pd.DataFrame(rows).sort_values("AIC")
fit_table

In [ ]:
# Step 3: Compare empirical return period points vs BestFit model curves
vals = np.array([df.ExactSeries[i].Value for i in range(df.ExactSeries.Count)], dtype=float)
p_plot = np.array([df.ExactSeries[i].PlottingPosition for i in range(df.ExactSeries.Count)], dtype=float)
# In this build plotting positions behave as exceedance probabilities for this series orientation.
T_emp = 1.0 / np.clip(p_plot, 1e-9, 1 - 1e-9)
ordr = np.argsort(T_emp)
T_emp, vals = T_emp[ordr], vals[ordr]

T_grid = np.logspace(np.log10(1.05), np.log10(200), 300)
F_grid = 1.0 - 1.0 / T_grid

plt.figure(figsize=(8,5))
plt.scatter(T_emp, vals, c='black', s=20, label='Empirical')
for name, m in results.items():
    q = np.array([float(m.Distribution.InverseCDF(float(np.clip(f, 1e-6, 1-1e-6)))) for f in F_grid])
    plt.plot(T_grid, q, lw=2, label=name)

plt.xscale('log')
plt.xlabel('Return period, T')
plt.ylabel('Peak flow')
plt.title('BestFit Distribution Curves')
plt.grid(alpha=0.3, which='both')
plt.legend()
plt.tight_layout()

## Summary and Exercises

You used BestFit `UnivariateDistribution` models directly for likelihood-based fitting and model comparison.

Recommended exercises:
1. Add `LogPearsonTypeIII` and compare its AIC/BIC against GEV.
2. Replace synthetic peaks with a real gage record and re-run.
3. Add parameter sensitivity plots by perturbing scale/shape and tracking `LogLikelihood`.